In [2]:
import numpy as np
np.random.seed(42)

conv_weights = np.random.randn(8, 3, 3, 3).astype(np.float32)

conv_weights[0] *= 0.1
conv_weights[1] *= 0.5
conv_weights[6] *= 5.0
conv_weights[7] *= 10.0

In [3]:
print(conv_weights.shape)

(8, 3, 3, 3)


In [4]:
def per_tensor_quantize(tensor):

    scale = np.max(np.abs(tensor)) / 127

    quantized = np.round(tensor / scale)

    quantized = np.clip(quantized, -127, 127)

    quantized = quantized.astype(np.int8)

    dequantized = quantized * scale

    return quantized, dequantized, scale

In [5]:
def per_channel_quantize(tensor):

    quantized = np.zeros_like(tensor, dtype=np.int8)

    dequantized = np.zeros_like(tensor, dtype=np.float32)

    scales = []

    for c in range(tensor.shape[0]):

        scale = np.max(np.abs(tensor[c])) / 127

        if scale == 0:
            scale = 1.0

        q = np.round(tensor[c] / scale)

        q = np.clip(q, -127, 127)

        q = q.astype(np.int8)

        dq = q * scale

        quantized[c] = q
        dequantized[c] = dq

        scales.append(scale)

    return quantized, dequantized, np.array(scales)

In [6]:
q_tensor, dq_tensor, tensor_scale = per_tensor_quantize(conv_weights)

q_channel, dq_channel, channel_scales = per_channel_quantize(conv_weights)

In [7]:
print("\n===== Channel-wise Comparison =====")

total_tensor_mae = []
total_channel_mae = []

for c in range(conv_weights.shape[0]):

    tensor_mae = np.mean(
        np.abs(conv_weights[c] - dq_tensor[c])
    )

    channel_mae = np.mean(
        np.abs(conv_weights[c] - dq_channel[c])
    )

    total_tensor_mae.append(tensor_mae)
    total_channel_mae.append(channel_mae)

    print("\nChannel", c)

    print("Range:",
          np.min(conv_weights[c]),
          "to",
          np.max(conv_weights[c]))

    print("Per-Tensor Scale:", tensor_scale)

    print("Per-Tensor MAE:", tensor_mae)

    print("Per-Channel Scale:", channel_scales[c])

    print("Per-Channel MAE:", channel_mae)

    if channel_mae < tensor_mae:
        print("Better: Per-Channel")
    elif tensor_mae < channel_mae:
        print("Better: Per-Tensor")
    else:
        print("Both Equal")


===== Channel-wise Comparison =====

Channel 0
Range: -0.19132803 to 0.15792128
Per-Tensor Scale: 0.30336466
Per-Tensor MAE: 0.07146436
Per-Channel Scale: 0.00150652
Per-Channel MAE: 0.00032642912
Better: Per-Channel

Channel 1
Range: -0.97983503 to 0.9261391
Per-Tensor Scale: 0.30336466
Per-Tensor MAE: 0.07256411
Per-Channel Scale: 0.0077152364
Per-Channel MAE: 0.0016606675
Better: Per-Channel

Channel 2
Range: -2.619745 to 1.5646436
Per-Tensor Scale: 0.30336466
Per-Tensor MAE: 0.072174944
Per-Channel Scale: 0.020627914
Per-Channel MAE: 0.005372616
Better: Per-Channel

Channel 3
Range: -1.4635149 to 1.8861859
Per-Tensor Scale: 0.30336466
Per-Tensor MAE: 0.071593724
Per-Channel Scale: 0.014851857
Per-Channel MAE: 0.0037305246
Better: Per-Channel

Channel 4
Range: -1.9187713 to 2.463242
Per-Tensor Scale: 0.30336466
Per-Tensor MAE: 0.07057091
Per-Channel Scale: 0.019395607
Per-Channel MAE: 0.003995452
Better: Per-Channel

Channel 5
Range: -1.6074833 to 1.8657745
Per-Tensor Scale: 0.3033

In [8]:
print("\n===========================")

print("Average Per-Tensor MAE:",
      np.mean(total_tensor_mae))

print("Average Per-Channel MAE:",
      np.mean(total_channel_mae))


Average Per-Tensor MAE: 0.07114315
Average Per-Channel MAE: 0.013842214


## Observations

- Per-tensor quantization uses a single scale for the entire weight tensor.
- Per-channel quantization computes a separate scale for each output channel.
- Channels with different value ranges benefit from independent scales.
- Per-channel quantization generally produced lower MAE than per-tensor quantization.
- Modern deep learning frameworks commonly use per-channel quantization for convolution weights because it preserves accuracy better.